# Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Load Data

In [2]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

# Dataset Overview

In [3]:
train_df.describe()

,id,Age,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Work/Study Hours,Financial Stress,Depression
count,140700.000000,140700.000000,27897.000000,112782.000000,27898.000000,27897.000000,112790.000000,140700.000000,140696.000000,140700.000000
mean,70349.500000,40.388621,3.142273,2.998998,7.658636,2.944940,2.974404,6.252679,2.988983,0.181713
std,40616.735775,12.384099,1.380457,1.405771,1.464466,1.360197,1.416078,3.853615,1.413633,0.385609
min,0.000000,18.000000,1.000000,1.000000,5.030000,1.000000,1.000000,0.000000,1.000000,0.000000
25%,35174.750000,29.000000,2.000000,2.000000,6.290000,2.000000,2.000000,3.000000,2.000000,0.000000
50%,70349.500000,42.000000,3.000000,3.000000,7.770000,3.000000,3.000000,6.000000,3.000000,0.000000
75%,105524.250000,51.000000,4.000000,4.000000,8.920000,4.000000,4.000000,10.000000,4.000000,0.000000
max,140699.000000,60.000000,5.000000,5.000000,10.000000,5.000000,5.000000,12.000000,5.000000,1.000000


In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140700 entries, 0 to 140699
Data columns (total 20 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   id                                     140700 non-null  int64  
 1   Name                                   140700 non-null  object 
 2   Gender                                 140700 non-null  object 
 3   Age                                    140700 non-null  float64
 4   City                                   140700 non-null  object 
 5   Working Professional or Student        140700 non-null  object 
 6   Profession                             104070 non-null  object 
 7   Academic Pressure                      27897 non-null   float64
 8   Work Pressure                          112782 non-null  float64
 9   CGPA                                   27898 non-null   float64
 10  Study Satisfaction                     27897 non-null   

In [5]:
numerical_vars = ['Age', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction',
                  'Job Satisfaction', 'Work/Study Hours', 'Financial Stress', 'Depression']

categorical_vars = ['Gender', 'Working Professional or Student', 'Sleep Duration', 'Dietary Habits', 
                    'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

In [6]:
train_df = train_df[numerical_vars+categorical_vars]

# Filter relevant columns

In [7]:
def identify_cat_above30(series):
    counts = series.value_counts()
    return list(counts[counts>=30].index)

In [8]:
levels_to_keep = train_df[categorical_vars].apply(identify_cat_above30, axis=0)
levels_to_keep

Gender                                                                      [Male, Female]
Working Professional or Student                            [Working Professional, Student]
Sleep Duration                           [Less than 5 hours, 7-8 hours, More than 8 hou...
Dietary Habits                                              [Moderate, Unhealthy, Healthy]
Have you ever had suicidal thoughts ?                                            [No, Yes]
Family History of Mental Illness                                                 [No, Yes]
dtype: object

In [9]:
for var in categorical_vars:
    train_df = train_df.loc[train_df[var].isin(levels_to_keep[var])]

# Handling Missing Values

In [10]:
total = train_df.isnull().sum().sort_values(ascending=False)

percent = (train_df.isnull().sum()/train_df.isnull().count()).sort_values(ascending=False)

missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Missing Percent'])

In [11]:
missing_data['Missing Percent'] = missing_data['Missing Percent'].apply(lambda x: x*100)
missing_data.loc[missing_data['Missing Percent'] > 10][:10]

,Total,Missing Percent
Academic Pressure,112727,80.179097
Study Satisfaction,112727,80.179097
CGPA,112726,80.178386
Work Pressure,27888,19.835839
Job Satisfaction,27880,19.830149


# Check for missing data and split by group (Student/Working Professional)

In [12]:
students_df = train_df[train_df['Working Professional or Student'] == 'Student']
working_professionals_df = train_df[train_df['Working Professional or Student'] == 'Working Professional']

# Handle missing values in the "Student" dataset

In [13]:
columns_to_drop = ['Work Pressure', 'Job Satisfaction']
students_df = students_df.drop(columns_to_drop, axis=1)

In [14]:
numerical_vars_students = ['Age', 'Academic Pressure', 'CGPA', 'Study Satisfaction',
                           'Work/Study Hours', 'Financial Stress']

categorical_vars_students = ['Gender', 'Working Professional or Student', 'Sleep Duration', 'Dietary Habits', 
                             'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

In [15]:
columns_to_drop = ['Academic Pressure', 'CGPA', 'Study Satisfaction']
working_professionals_df = working_professionals_df.drop(columns_to_drop, axis=1)

In [16]:
numerical_vars_working = ['Age', 'Work Pressure', 'Job Satisfaction',
                          'Work/Study Hours', 'Financial Stress']

categorical_vars_working = ['Gender', 'Working Professional or Student', 'Sleep Duration', 'Dietary Habits', 
                            'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

In [17]:
students_df_train = students_df[numerical_vars_students+categorical_vars_students]

In [18]:
working_professionals_df_train = working_professionals_df[numerical_vars_working+categorical_vars_working]

# Handle missing values for numerical variables using SimpleImputer

In [19]:
total = students_df_train.isnull().sum().sort_values(ascending=False)

percent = (students_df_train.isnull().sum()/students_df_train.isnull().count()).sort_values(ascending=False)

missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Missing Percent'])

In [20]:
missing_data

,Total,Missing Percent
Study Satisfaction,10,0.000359
Academic Pressure,9,0.000323
CGPA,9,0.000323
Financial Stress,3,0.000108
Age,0,0.000000
Work/Study Hours,0,0.000000
Gender,0,0.000000
Working Professional or Student,0,0.000000
Sleep Duration,0,0.000000
Dietary Habits,0,0.000000


In [21]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')


students_df_train.loc[:, numerical_vars_students] = imputer.fit_transform(students_df_train[numerical_vars_students])

In [22]:
total = students_df_train.isnull().sum().sort_values(ascending=False)

percent = (students_df_train.isnull().sum()/students_df_train.isnull().count()).sort_values(ascending=False)

missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Missing Percent'])

In [23]:
missing_data

,Total,Missing Percent
Age,0,0.0
Academic Pressure,0,0.0
CGPA,0,0.0
Study Satisfaction,0,0.0
Work/Study Hours,0,0.0
Financial Stress,0,0.0
Gender,0,0.0
Working Professional or Student,0,0.0
Sleep Duration,0,0.0
Dietary Habits,0,0.0


# Handle missing values in the "Working Professional" dataset

In [24]:
total = working_professionals_df_train.isnull().sum().sort_values(ascending=False)

percent = (working_professionals_df_train.isnull().sum()/working_professionals_df_train.isnull().count()).sort_values(ascending=False)

missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Missing Percent'])

In [25]:
missing_data

,Total,Missing Percent
Work Pressure,20,0.000177
Job Satisfaction,17,0.000151
Financial Stress,1,0.000009
Age,0,0.000000
Work/Study Hours,0,0.000000
Gender,0,0.000000
Working Professional or Student,0,0.000000
Sleep Duration,0,0.000000
Dietary Habits,0,0.000000
Have you ever had suicidal thoughts ?,0,0.000000


In [26]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')

working_professionals_df_train.loc[:, numerical_vars_working] = imputer.fit_transform(working_professionals_df_train[numerical_vars_working])

In [27]:
total = working_professionals_df_train.isnull().sum().sort_values(ascending=False)

percent = (working_professionals_df_train.isnull().sum()/working_professionals_df_train.isnull().count()).sort_values(ascending=False)

missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Missing Percent'])

In [28]:
missing_data

,Total,Missing Percent
Age,0,0.0
Work Pressure,0,0.0
Job Satisfaction,0,0.0
Work/Study Hours,0,0.0
Financial Stress,0,0.0
Gender,0,0.0
Working Professional or Student,0,0.0
Sleep Duration,0,0.0
Dietary Habits,0,0.0
Have you ever had suicidal thoughts ?,0,0.0


# Split the data into features and labels

In [29]:
X_students = students_df_train
y_students = students_df['Depression']

In [30]:
X_working = working_professionals_df_train
y_working = working_professionals_df['Depression']

# Encoding categorical variables

In [31]:
from sklearn.preprocessing import LabelEncoder

for col in categorical_vars_students:
    le = LabelEncoder()
    X_students.loc[:, col] = le.fit_transform(X_students[col])

In [32]:
for col in categorical_vars_working:
    le = LabelEncoder()
    X_working.loc[:, col] = le.fit_transform(X_working[col])

# Standardize the features

In [33]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

In [34]:
X_students = scaler.fit_transform(X_students)

In [35]:
X_working = scaler.fit_transform(X_working)

# Handle class imbalance using SMOTE

In [36]:
count_class = y_students.value_counts()
count_class

Depression
1    16319
0    11552
Name: count, dtype: int64

In [37]:
count_class = y_working.value_counts()
count_class

Depression
0    103496
1      9227
Name: count, dtype: int64

In [38]:
from imblearn.over_sampling import SMOTE

In [39]:
smote = SMOTE(sampling_strategy='minority')
X_students, y_students = smote.fit_resample(X_students, y_students)

In [40]:
X_working, y_working = smote.fit_resample(X_working, y_working)

# Split the data into training and testing sets

In [41]:
X_train_students, X_test_students, y_train_students, y_test_students = train_test_split(
    X_students, y_students, test_size=0.2, random_state=42
)

In [42]:
X_train_working, X_test_working, y_train_working, y_test_working = train_test_split(
    X_working, y_working, test_size=0.2, random_state=42
)

# Model 1: Random Forest

In [43]:
print("=== Random Forest Model ===")
rf_model_student = RandomForestClassifier(n_estimators=100, random_state=42)

=== Random Forest Model ===


In [44]:
cv_scores_rf = cross_val_score(rf_model_student, X_train_students, y_train_students, cv=5, scoring='accuracy')
print("Cross-Validation Scores (Random Forest):", cv_scores_rf)
print("Mean CV Accuracy (Random Forest):", np.mean(cv_scores_rf))
print("Standard Deviation CV Accuracy (Random Forest):", np.std(cv_scores_rf))

Cross-Validation Scores (Random Forest): [0.86001532 0.85561088 0.8615473  0.86078131 0.86556875]
Mean CV Accuracy (Random Forest): 0.8607047108387592
Standard Deviation CV Accuracy (Random Forest): 0.0031873838792130325


In [45]:
rf_model_student.fit(X_train_students, y_train_students)

RandomForestClassifier(random_state=42)

In [46]:
y_pred_rf = rf_model_student.predict(X_test_students)
print("\nTest Set Evaluation Student (Random Forest)")
print(classification_report(y_test_students, y_pred_rf))
print("Test Accuracy (Random Forest):", accuracy_score(y_test_students, y_pred_rf))


Test Set Evaluation Student (Random Forest)
              precision    recall  f1-score   support

           0       0.86      0.85      0.86      3255
           1       0.86      0.86      0.86      3273

    accuracy                           0.86      6528
   macro avg       0.86      0.86      0.86      6528
weighted avg       0.86      0.86      0.86      6528

Test Accuracy (Random Forest): 0.8589154411764706


In [47]:
rf_model_working = RandomForestClassifier(n_estimators=100, random_state=42)

In [48]:
cv_scores_rf = cross_val_score(rf_model_working, X_train_working, y_train_working, cv=5, scoring='accuracy')
print("Cross-Validation Scores (Random Forest):", cv_scores_rf)
print("Mean CV Accuracy (Random Forest):", np.mean(cv_scores_rf))
print("Standard Deviation CV Accuracy (Random Forest):", np.std(cv_scores_rf))

Cross-Validation Scores (Random Forest): [0.97557293 0.97608623 0.97590507 0.97454556 0.97653844]
Mean CV Accuracy (Random Forest): 0.9757296481415663
Standard Deviation CV Accuracy (Random Forest): 0.0006691310510166426


In [49]:
rf_model_working.fit(X_train_working, y_train_working)

RandomForestClassifier(random_state=42)

In [50]:
y_pred_rf = rf_model_working.predict(X_test_working)
print("\nTest Set Evaluation Working Professional (Random Forest)")
print(classification_report(y_test_working, y_pred_rf))
print("Test Accuracy (Random Forest):", accuracy_score(y_test_working, y_pred_rf))


Test Set Evaluation Working Professional (Random Forest)
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     20810
           1       0.98      0.98      0.98     20589

    accuracy                           0.98     41399
   macro avg       0.98      0.98      0.98     41399
weighted avg       0.98      0.98      0.98     41399

Test Accuracy (Random Forest): 0.9787676030822


# Model 2: XGBoost

In [51]:
param_grid = {
    'n_estimators': [50, 100, 200],  
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

In [52]:
print("=== XGBoost Model ===")
xgb_model_student = XGBClassifier(random_state=42, eval_metric='logloss')

=== XGBoost Model ===


In [53]:
grid_search = GridSearchCV(estimator=xgb_model_student,
                           param_grid=param_grid,
                           cv=3,  # 3-Fold Cross Validation
                           scoring='accuracy',
                           verbose=1,
                           n_jobs=-1)

In [54]:
grid_search.fit(X_train_students, y_train_students)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


GridSearchCV(cv=3,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='logloss', feature_types=None,
                                     gamma=None, grow_policy=None,
                                     importance_type=None,
                                     interaction_constraints=None,
                                     learning_rate=...
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=None,
                                     n_jobs=None, num_parallel_tree=None,
                                     random_state=42, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 1.0],
                         'learning_rate': [0.01, 0.1, 0.2],
                         'max_depth': [3, 5, 7], 'n_estimators': [50, 100, 200],
                         'subsample': [0.8, 1.0]},
             scoring='accuracy', verbose=1)

In [55]:
print("Best Parameters:", grid_search.best_params_)

Best Parameters: {'colsample_bytree': 1.0, 'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200, 'subsample': 1.0}


In [56]:
best_xgb_model_student = grid_search.best_estimator_

In [57]:
y_pred_best_xgb = best_xgb_model_student.predict(X_test_students)

In [58]:
print("\nTest Set Evaluation Student (Best XGBoost)")
print(classification_report(y_test_students, y_pred_best_xgb))
print("Test Accuracy (Best XGBoost):", accuracy_score(y_test_students, y_pred_best_xgb))


Test Set Evaluation Student (Best XGBoost)
              precision    recall  f1-score   support

           0       0.87      0.86      0.86      3255
           1       0.86      0.88      0.87      3273

    accuracy                           0.87      6528
   macro avg       0.87      0.87      0.87      6528
weighted avg       0.87      0.87      0.87      6528

Test Accuracy (Best XGBoost): 0.8661151960784313


In [ ]:
grid_search.fit(X_train_working, y_train_working)

Fitting 3 folds for each of 108 candidates, totalling 324 fits


In [ ]:
print("Best Parameters:", grid_search.best_params_)

In [ ]:
best_xgb_model_working = grid_search.best_estimator_

In [ ]:
y_pred_best_xgb = best_xgb_model_working.predict(X_test_working)
print("\nTest Set Evaluation Working (Best XGBoost)")
print(classification_report(y_test_working, y_pred_best_xgb))
print("Test Accuracy (Best XGBoost):", accuracy_score(y_test_working, y_pred_best_xgb))

# Model 3: Neural Network (Keras)

In [ ]:
model_student = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_students.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

In [ ]:
model_student.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
history_student = model_student.fit(
    X_train_students, y_train_students,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
test_loss, test_accuracy = model_student.evaluate(X_train_students, y_train_students)
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
y_pred_students_nn = model_student.predict(X_test_students)
y_pred_students_nn = (y_pred_students_nn > 0.5).astype(int)
print("\nTest Set Evaluation Student (Neural Network)")
print(classification_report(y_test_students, y_pred_students_nn))
print("Test Accuracy (Neural Network):", accuracy_score(y_test_students, y_pred_students_nn))

In [ ]:
model_working = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_working.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Output layer for binary classification
])

In [ ]:
model_working.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
history_working = model_working.fit(
    X_train_working, y_train_working,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
test_loss, test_accuracy = model_working.evaluate(X_train_working, y_train_working)
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
y_pred_working_nn = model_working.predict(X_test_working)
y_pred_working_nn = (y_pred_working_nn > 0.5).astype(int)
print("\nTest Set Evaluation Working Professional (Neural Network)")
print(classification_report(y_test_working, y_pred_working_nn))
print("Test Accuracy (Neural Network):", accuracy_score(y_test_working, y_pred_working_nn))

# Test Dataset

In [ ]:
students_test = test_df[test_df['Working Professional or Student'] == 'Student']
working_professionals_test = test_df[test_df['Working Professional or Student'] == 'Working Professional']

In [ ]:
columns_to_drop = ['Work Pressure', 'Job Satisfaction']
students_df_test = students_test.drop(columns_to_drop, axis=1)

In [ ]:
num_vars_students_test = ['Age', 'Academic Pressure', 'CGPA', 'Study Satisfaction',
                           'Work/Study Hours', 'Financial Stress']

cat_vars_students_test = ['Gender', 'Working Professional or Student', 'Sleep Duration', 'Dietary Habits', 
                          'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

In [ ]:
students_df_test = students_df_test[num_vars_students_test+cat_vars_students_test]

In [ ]:
columns_to_drop = ['Academic Pressure', 'CGPA', 'Study Satisfaction']
working_professionals_df_test = working_professionals_test.drop(columns_to_drop, axis=1)

In [ ]:
num_vars_working_test = ['Age', 'Work Pressure', 'Job Satisfaction',
                          'Work/Study Hours', 'Financial Stress']

cat_vars_working_test = ['Gender', 'Working Professional or Student', 'Sleep Duration', 'Dietary Habits', 
                            'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']

In [ ]:
working_professionals_df_test = working_professionals_df_test[num_vars_working_test+cat_vars_working_test]

In [ ]:
imputer = SimpleImputer(strategy='mean')
students_df_test[num_vars_students_test] = imputer.fit_transform(students_df_test[num_vars_students_test])
working_professionals_df_test[num_vars_working_test] = imputer.fit_transform(working_professionals_df_test[num_vars_working_test])

In [ ]:
for col in cat_vars_students_test:
    le = LabelEncoder()
    students_df_test[col] = le.fit_transform(students_df_test[col].astype(str))

In [ ]:
for col in cat_vars_working_test:
    le = LabelEncoder()
    working_professionals_df_test[col] = le.fit_transform(working_professionals_df_test[col].astype(str))

In [ ]:
scaler = StandardScaler()

students_df_test_scaled = scaler.fit_transform(students_df_test)
working_professionals_df_test_scaled = scaler.fit_transform(working_professionals_df_test)

# Test RF

In [ ]:
y_pred_students = rf_model_student.predict(students_df_test_scaled)

In [ ]:
y_pred_working = rf_model_working.predict(working_professionals_df_test_scaled)

In [ ]:
students_test.loc[:, 'Depression'] = y_pred_students

In [ ]:
working_professionals_test.loc[:, 'Depression'] = y_pred_working

In [ ]:
students_submission_rf = students_test[['id', 'Depression']]
working_submission_rf = working_professionals_test[['id', 'Depression']]

submission_rf = pd.concat([students_submission_rf, working_submission_rf], axis=0)

In [ ]:
submission_rf

In [ ]:
submission_rf.to_csv('../data/submission-rf-smote.csv', index=False)

# Test XGB

In [ ]:
y_pred_students_xgb = best_xgb_model_student.predict(students_df_test_scaled)

In [ ]:
y_pred_working_xgb = best_xgb_model_working.predict(working_professionals_df_test_scaled)

In [ ]:
students_test.loc[:, 'Depression'] = y_pred_students_xgb

In [ ]:
working_professionals_test.loc[:, 'Depression'] = y_pred_working_xgb

In [ ]:
students_submission_xgb = students_test[['id', 'Depression']]
working_submission_xgb = working_professionals_test[['id', 'Depression']]

submission_xgb = pd.concat([students_submission_xgb, working_submission_xgb], axis=0)

In [ ]:
submission_xgb

In [ ]:
submission_xgb.to_csv('../data/submission-xgb-smote.csv', index=False)

# Test Neural Network

In [ ]:
y_pred_students_nn = model_student.predict(students_df_test_scaled)
y_pred_students_nn = (y_pred_students_nn > 0.5).astype(int)

In [ ]:
y_pred_working_nn = model_working.predict(working_professionals_df_test_scaled)
y_pred_working_nn = (y_pred_working_nn > 0.5).astype(int)

In [ ]:
students_test.loc[:, 'Depression'] = y_pred_students_nn

In [ ]:
working_professionals_test.loc[:, 'Depression'] = y_pred_working_

In [ ]:
students_submission_nn = students_test[['id', 'Depression']]
working_submission_nn = working_professionals_test[['id', 'Depression']]

submission_nn = pd.concat([students_submission_nn, working_submission_nn], axis=0)

In [ ]:
submission_nn

In [ ]:
submission_nn.to_csv('../data/submission-nn-smote.csv', index=False)